# Notebook 1: Dataset Engineering for LLM Fine-Tuning

**Goal:** Learn to load, clean, transform, and visualize instruction-tuning datasets. By the end you will build a mini ShareGPT-format dataset from scratch.

**Topics covered:**
- Loading datasets from Hugging Face Hub
- Format conversion: Alpaca, ShareGPT, Qwen Chat
- Data cleaning: deduplication, quality filtering, format unification
- Train/validation splitting with stratification
- Data distribution visualization
- Human inspection & quality sampling
- Building a custom dataset with `datasets.Dataset.from_dict()`

In [ ]:
# Cell 1: Install dependencies
!pip install datasets transformers matplotlib scikit-learn pandas tqdm pyarrow -q

In [ ]:
# Cell 2: Imports and setup
import json
import random
import hashlib
import re
from collections import Counter
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

from datasets import Dataset, DatasetDict, load_dataset, concatenate_datasets
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (10, 6)

print("All imports successful!")

## Part 1: Loading Datasets from Hugging Face Hub

We will load three popular instruction-tuning datasets and examine their formats.

In [ ]:
# Cell 3: Load datasets from Hugging Face
# Using small subsets to keep things fast

# Alpaca format: (instruction, input, output) tuples
print("Loading Alpaca dataset...")
alpaca = load_dataset("tatsu-lab/alpaca", split="train[:500]")
print(f"Alpaca columns: {alpaca.column_names}")
print(f"Alpaca rows: {len(alpaca)}")

# Dolly dataset: instruction/context/response/category
print("\nLoading Dolly dataset...")
dolly = load_dataset("databricks/databricks-dolly-15k", split="train[:500]")
print(f"Dolly columns: {dolly.column_names}")
print(f"Dolly rows: {len(dolly)}")

# OpenOrca: ShareGPT-style conversations
print("\nLoading OpenOrca (ShareGPT-style) dataset...")
openorca = load_dataset("Open-Orca/OpenOrca", split="train[:200]")
print(f"OpenOrca columns: {openorca.column_names}")
print(f"OpenOrca rows: {len(openorca)}")

In [ ]:
# Cell 4: Inspect raw data samples

print("=" * 60)
print("ALPACA SAMPLE")
print("=" * 60)
sample = alpaca[0]
for k, v in sample.items():
    print(f"\n[{k}]:\n{v[:300] if isinstance(v, str) and len(v) > 300 else v}")

print("\n" + "=" * 60)
print("DOLLY SAMPLE")
print("=" * 60)
sample = dolly[0]
for k, v in sample.items():
    print(f"\n[{k}]:\n{v[:300] if isinstance(v, str) and len(v) > 300 else v}")

print("\n" + "=" * 60)
print("OPENORCA SAMPLE (system_prompt + question + response)")
print("=" * 60)
sample = openorca[0]
for k, v in sample.items():
    val_str = str(v)[:300]
    print(f"\n[{k}]:\n{val_str}")

## Part 2: Format Converters — Alpaca, ShareGPT, Qwen Chat

Different models expect different conversation formats. Here we build converters for three common formats:

In [ ]:
# Cell 5: Format conversion functions

def alpaca_to_sharegpt(example):
    """
    Convert Alpaca format (instruction, input, output) to ShareGPT format.
    ShareGPT format: {"conversations": [{"from": "human/gpt", "value": "..."}]}
    """
    conv = []
    prompt = example["instruction"]
    if example.get("input") and example["input"].strip():
        prompt += f"\n\n{example['input']}"
    conv.append({"from": "human", "value": prompt})
    conv.append({"from": "gpt", "value": example["output"]})
    return {"conversations": conv}


def sharegpt_to_qwen_chat(example):
    """
    Convert ShareGPT format to Qwen Chat format.
    Qwen Chat: {"messages": [{"role": "user/assistant", "content": "..."}]}
    Adds a system message at the start.
    """
    messages = []
    messages.append({"role": "system", "content": "You are a helpful assistant."})
    for turn in example["conversations"]:
        role = "user" if turn["from"] == "human" else "assistant"
        messages.append({"role": role, "content": turn["value"]})
    return {"messages": messages}


def dolly_to_sharegpt(example):
    """Convert Dolly format to ShareGPT format."""
    conv = []
    prompt = example["instruction"]
    if example.get("context") and example["context"].strip():
        prompt += f"\n\nContext: {example['context']}"
    conv.append({"from": "human", "value": prompt})
    conv.append({"from": "gpt", "value": example["response"]})
    return {"conversations": conv}


print("Format converter functions defined.")

In [ ]:
# Cell 6: Apply converters and compare formats

# Convert Alpaca -> ShareGPT
alpaca_sg = alpaca.map(alpaca_to_sharegpt, remove_columns=alpaca.column_names)

# Convert Dolly -> ShareGPT
dolly_sg = dolly.map(dolly_to_sharegpt, remove_columns=dolly.column_names)

# Convert ShareGPT -> Qwen Chat
alpaca_qwen = alpaca_sg.map(sharegpt_to_qwen_chat, remove_columns=["conversations"])

print("All conversions complete.")
print(f"Alpaca->ShareGPT: {len(alpaca_sg)} examples")
print(f"Dolly->ShareGPT: {len(dolly_sg)} examples")
print(f"Alpaca->QwenChat: {len(alpaca_qwen)} examples")

print("\n--- ShareGPT format example ---")
print(json.dumps(alpaca_sg[0]["conversations"], indent=2))

print("\n--- Qwen Chat format example ---")
print(json.dumps(alpaca_qwen[0]["messages"], indent=2))

In [ ]:
# Cell 7: Format unification — merge all into ShareGPT
# This is a common pattern: pipeline inputs from multiple sources, unify to one format

# Combine Alpaca and Dolly ShareGPT conversions
unified_dataset = concatenate_datasets([alpaca_sg, dolly_sg])
print(f"Unified dataset: {len(unified_dataset)} examples")

# Now add a unique ID to each conversation
def add_id(example, idx):
    example["id"] = f"conv_{idx:06d}"
    return example

unified_dataset = unified_dataset.map(add_id, with_indices=True)
print(f"Added IDs. First ID: {unified_dataset[0]['id']}")

## Part 3: Data Cleaning Pipeline

Real-world datasets contain duplicates, low-quality examples, and formatting issues. We build a cleaning pipeline with:
1. Deduplication (MinHash + exact match)
2. Quality filtering (length, entropy, language detection)
3. Format validation

In [ ]:
# Cell 8: Quality filtering — length, word count, entropy

def compute_conversation_stats(example):
    """Compute quality metrics for a conversation."""
    conv = example["conversations"]
    human_turns = [t for t in conv if t["from"] == "human"]
    gpt_turns = [t for t in conv if t["from"] == "gpt"]

    human_text = " ".join([t["value"] for t in human_turns])
    gpt_text = " ".join([t["value"] for t in gpt_turns])

    total_chars = sum(len(t["value"]) for t in conv)
    total_words = sum(len(t["value"].split()) for t in conv)
    num_turns = len(conv)

    # Approximate entropy (character-level Shannon entropy)
    all_text = "".join([t["value"] for t in conv])
    char_counts = Counter(all_text.lower())
    total = sum(char_counts.values())
    entropy = -sum((c / total) * np.log2(c / total) for c in char_counts.values()) if total > 0 else 0

    return {
        "total_chars": total_chars,
        "total_words": total_words,
        "num_turns": num_turns,
        "char_entropy": round(entropy, 3),
        "has_content": total_words > 10,
    }

# Compute stats
unified_dataset = unified_dataset.map(compute_conversation_stats)

# Show stats distribution
df = unified_dataset.to_pandas()
print("Quality stats summary:")
print(df[["total_chars", "total_words", "num_turns", "char_entropy"]].describe())

In [ ]:
# Cell 9: Apply quality filters

def quality_filter(example):
    """Keep only high-quality examples based on heuristics."""
    # Minimum content length
    if example["total_words"] < 20:
        return False
    # Maximum length (avoid excessively long examples that break context windows)
    if example["total_chars"] > 8000:
        return False
    # Entropy too low suggests repetitive / degenerate text
    if example["char_entropy"] < 3.0:
        return False
    # Must have at least 2 turns (human + gpt)
    if example["num_turns"] < 2:
        return False
    return True

before = len(unified_dataset)
unified_dataset = unified_dataset.filter(quality_filter)
after = len(unified_dataset)
print(f"Quality filtering: {before} -> {after} examples ({before - after} removed)")

In [ ]:
# Cell 10: Deduplication — exact match & MinHash demo

def compute_text_hash(example):
    """Compute a hash of the full conversation text for exact dedup."""
    text = json.dumps(example["conversations"], sort_keys=True)
    example["text_hash"] = hashlib.sha256(text.encode()).hexdigest()
    return example

unified_dataset = unified_dataset.map(compute_text_hash)

# Exact dedup via hash
def dedup_exact(dataset):
    seen = set()
    keep_indices = []
    for i, example in enumerate(dataset):
        h = example["text_hash"]
        if h not in seen:
            seen.add(h)
            keep_indices.append(i)
    return dataset.select(keep_indices)

before = len(unified_dataset)
unified_dataset = dedup_exact(unified_dataset)
print(f"Exact dedup: {before} -> {len(unified_dataset)}")

# --- MinHash demo for near-duplicate detection ---
print("\n--- MinHash Near-Duplicate Demo ---")

# Create shingle sets for a few examples
def get_shingles(text, k=3):
    """Convert text to a set of character k-shingles."""
    return set(text[i:i+k] for i in range(len(text) - k + 1))

# Simulate MinHash by hashing shingles (simplified demo)
def minhash_signature(shingle_set, num_hashes=128):
    """Simplified MinHash signature."""
    sig = []
    for i in range(num_hashes):
        min_val = float('inf')
        for s in shingle_set:
            h = hash((s, i)) % (2**32)
            if h < min_val:
                min_val = h
        sig.append(min_val)
    return sig

def jaccard_estimate(sig1, sig2):
    """Estimate Jaccard similarity from MinHash signatures."""
    matches = sum(1 for a, b in zip(sig1, sig2) if a == b)
    return matches / len(sig1)

# Demo on 3 random examples
indices = random.sample(range(len(unified_dataset)), min(3, len(unified_dataset)))
for i in indices:
    text_a = json.dumps(unified_dataset[i]["conversations"], sort_keys=True)
    shingles = get_shingles(text_a, k=3)
    sig = minhash_signature(shingles, num_hashes=64)
    print(f"  conv_{i}: {len(shingles)} shingles, MinHash sig[:5] = {sig[:5]}")

print("\nMinHash is used in production with libraries like datasketch for scalable near-dedup.")
print(f"Final dataset size after cleaning: {len(unified_dataset)} examples")

## Part 4: Train/Validation Split with Stratification

We stratify by conversation length to ensure both splits have similar length distributions.

In [ ]:
# Cell 11: Stratified train/val split

# Create a categorical length bucket for stratification
def get_length_bucket(n_words):
    if n_words < 50:
        return "short"
    elif n_words < 150:
        return "medium"
    else:
        return "long"

df = unified_dataset.to_pandas()
df["length_bucket"] = df["total_words"].apply(get_length_bucket)

# Stratified split
train_df, val_df = train_test_split(
    df,
    test_size=0.15,
    stratify=df["length_bucket"],
    random_state=SEED,
)

# Convert back to Dataset
train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_df, preserve_index=False)

final_dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
})

print(f"Train: {len(train_dataset)} examples")
print(f"Validation: {len(val_dataset)} examples")

# Verify stratification
print("\nStratification check — length bucket distributions:")
print("Train:", train_df["length_bucket"].value_counts(normalize=True).to_dict())
print("Val:  ", val_df["length_bucket"].value_counts(normalize=True).to_dict())

## Part 5: Data Distribution Visualization

In [ ]:
# Cell 12: Visualize distributions

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Word count histogram
ax = axes[0]
ax.hist(train_df["total_words"], bins=30, alpha=0.7, label="Train", color="steelblue")
ax.hist(val_df["total_words"], bins=30, alpha=0.7, label="Validation", color="coral")
ax.set_xlabel("Word Count")
ax.set_ylabel("Frequency")
ax.set_title("Conversation Length Distribution")
ax.legend()

# 2. Character entropy histogram
ax = axes[1]
ax.hist(train_df["char_entropy"], bins=30, alpha=0.7, label="Train", color="steelblue")
ax.hist(val_df["char_entropy"], bins=30, alpha=0.7, label="Validation", color="coral")
ax.set_xlabel("Character Entropy")
ax.set_ylabel("Frequency")
ax.set_title("Text Entropy Distribution")
ax.legend()

# 3. Length bucket pie chart
ax = axes[2]
bucket_counts = train_df["length_bucket"].value_counts()
colors = ["#66b3ff", "#ffcc99", "#99ff99"]
ax.pie(bucket_counts.values, labels=bucket_counts.index, autopct="%1.1f%%",
       colors=colors, startangle=90)
ax.set_title("Length Categories (Train)")

plt.tight_layout()
plt.savefig("dataset_stats.png", dpi=100, bbox_inches="tight")
plt.show()
print("Chart saved to dataset_stats.png")

In [ ]:
# Cell 13: Category / topic distribution if available
# Dolly has a 'category' field — let's look at it

dolly_full = load_dataset("databricks/databricks-dolly-15k", split="train")
categories = dolly_full["category"]
cat_counts = Counter(categories)

fig, ax = plt.subplots(figsize=(10, 6))
top_cats = cat_counts.most_common(12)
cats, counts = zip(*top_cats)
bars = ax.bar(range(len(cats)), counts, color=plt.cm.viridis(np.linspace(0.2, 0.8, len(cats))))
ax.set_xticks(range(len(cats)))
ax.set_xticklabels(cats, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Number of Examples")
ax.set_title("Dolly Dataset — Top Categories")

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            str(count), ha="center", fontsize=8)

plt.tight_layout()
plt.savefig("dolly_categories.png", dpi=100, bbox_inches="tight")
plt.show()

## Part 6: Human Inspection Script

Randomly sample conversations and pretty-print them for manual quality review.

In [ ]:
# Cell 14: Pretty-print sampled conversations for human review

def pretty_print_conversation(example, index=None):
    """Print a conversation in a readable format for human inspection."""
    header = f"CONVERSATION {index}" if index is not None else "CONVERSATION"
    print("=" * 70)
    print(f"  {header}")
    if "id" in example:
        print(f"  ID: {example['id']}")
    print(f"  Words: {example.get('total_words', 'N/A')}  |  "
          f"Entropy: {example.get('char_entropy', 'N/A')}  |  "
          f"Turns: {example.get('num_turns', 'N/A')}")
    print("=" * 70)

    for turn in example["conversations"]:
        role = turn["from"].upper()
        text = turn["value"]
        # Truncate very long turns for readability
        if len(text) > 500:
            text = text[:500] + "... [truncated]"
        print(f"\n[{role}]")
        print(text)
    print("\n")

# Sample 5 random conversations
print("RANDOM SAMPLE FOR HUMAN INSPECTION\n")
num_samples = min(5, len(train_dataset))
sampled_indices = random.sample(range(len(train_dataset)), num_samples)
for i, idx in enumerate(sampled_indices):
    pretty_print_conversation(train_dataset[idx], index=idx)

In [ ]:
# Cell 15: Quality score card — flag potentially problematic examples

def quality_flags(example):
    """Return a list of quality flags for an example."""
    flags = []
    conv = example["conversations"]

    # Check for empty turns
    for i, turn in enumerate(conv):
        if not turn["value"].strip():
            flags.append(f"EMPTY_TURN_{i}")

    # Check for repeated substrings (potential template leakage)
    all_text = " ".join(t["value"] for t in conv)
    words = all_text.split()
    if len(words) > 0:
        unique_ratio = len(set(words)) / len(words)
        if unique_ratio < 0.5:
            flags.append("LOW_UNIQUE_WORD_RATIO")

    # Check if human text is much longer than assistant (unbalanced)
    human_len = sum(len(t["value"]) for t in conv if t["from"] == "human")
    gpt_len = sum(len(t["value"]) for t in conv if t["from"] == "gpt")
    if gpt_len > 0 and human_len / gpt_len > 5:
        flags.append("HUMAN_MUCH_LONGER")
    if human_len > 0 and gpt_len / human_len > 10:
        flags.append("GPT_MUCH_LONGER")

    # Check for code blocks leaking into non-code contexts
    code_indicators = ["```", "def ", "import ", "class ", "<?php"]
    for indicator in code_indicators:
        if indicator in all_text:
            flags.append("CONTAINS_CODE_BLOCK")
            break

    return flags if flags else ["OK"]

# Run quality flags on the validation set
flag_results = []
for example in val_dataset:
    flag_results.append(quality_flags(example))

all_flags = [f for flags in flag_results for f in flags]
flag_counts = Counter(all_flags)
print("Quality Flag Distribution (Validation Set):")
for flag, count in flag_counts.most_common():
    print(f"  {flag}: {count}")

## Part 7: Final Exercise — Build a Mini Customer-Service Dataset

Create 50-100 conversation examples in ShareGPT format for a "customer service" domain using `datasets.Dataset.from_dict()`.

In [ ]:
# Cell 16: Template-based synthetic customer service conversations

CUSTOMER_TEMPLATES = [
    # (user_question, agent_response)
    ("I need to return an item I bought last week. How do I do that?",
     "I'd be happy to help you with your return. Could you please provide me with your order number so I can look up your purchase? Our return window is 30 days from delivery."),
    ("My package says delivered but I haven't received it.",
     "I understand how concerning that must be. Let me help you track this down. First, please check with neighbors and around your property. If you still can't locate it after 24 hours, contact us again and we'll start a missing package investigation."),
    ("Can I change my shipping address after placing an order?",
     "Address changes depend on the order status. If your order hasn't shipped yet, we can usually update the address. Please share your order number and the new address, and I'll check if a change is still possible."),
    ("What's your refund policy?",
     "Our refund policy allows returns within 30 days of delivery for most items. Once we receive the returned item, refunds typically process within 5-10 business days to your original payment method. Some categories like personalized items may have different terms."),
    ("I received a damaged item. What should I do?",
     "I'm sorry to hear that. We take damaged deliveries very seriously. Please send us photos of the damaged item and packaging, along with your order number. We'll arrange a replacement or full refund right away."),
    ("How do I cancel my subscription?",
     "You can cancel your subscription anytime through your Account Settings under 'Manage Subscription', or I can help you do it right now. Would you like me to process the cancellation, or would you prefer to pause your subscription instead?"),
    ("Do you ship internationally?",
     "Yes, we ship to over 50 countries worldwide. International shipping costs and delivery times vary by destination. You can see the exact cost at checkout after entering your address. Please note that customs fees may apply."),
    ("I forgot my password. How do I reset it?",
     "No problem! Click the 'Forgot Password' link on the login page and enter your email address. We'll send you a password reset link within a few minutes. Make sure to check your spam folder if you don't see it."),
    ("Can I get a price adjustment for an item that went on sale?",
     "We offer price adjustments for items purchased within 7 days of a sale. Please provide your order number and I'll check your eligibility and process the difference back to your payment method."),
    ("My payment keeps getting declined. Help!",
     "I understand the frustration. Payment declines can happen for several reasons. Please verify that your billing address matches what your bank has on file, and that you have sufficient funds. You can also try using a different payment method or contact your bank directly."),
]

# Generate multi-turn variations
def generate_conversations(num_total=80):
    conversations = []
    for i in range(num_total):
        # Pick a base template
        base_q, base_a = random.choice(CUSTOMER_TEMPLATES)

        conv = [
            {"from": "human", "value": base_q},
            {"from": "gpt", "value": base_a},
        ]

        # 40% chance of a follow-up turn
        if random.random() < 0.4:
            followups = [
                ("Thank you! How long will this take?",
                 "You're welcome! Most requests are processed within 24 hours. You'll receive a confirmation email once it's complete. Is there anything else I can help with?"),
                ("Is there a fee for this?",
                 "No, there is no fee for this service. It's included as part of your customer benefits. Would you like me to go ahead and process this for you?"),
                ("Can you expedite this?",
                 "I can certainly add a priority flag to your case. While I can't guarantee faster processing, flagged cases are typically reviewed within 4-6 hours. Is there a specific deadline you're working with?"),
                ("What if it doesn't work?",
                 "If for any reason it doesn't resolve the issue, please reach back out to us and we'll escalate your case to our specialist team. We're committed to making this right. Is there anything else I can assist with today?"),
            ]
            f_q, f_a = random.choice(followups)
            conv.append({"from": "human", "value": f_q})
            conv.append({"from": "gpt", "value": f_a})

        conversations.append({
            "id": f"cs_{i:04d}",
            "conversations": conv,
            "domain": "customer_service",
        })
    return conversations

cs_conversations = generate_conversations(80)
print(f"Generated {len(cs_conversations)} customer-service conversations")

In [ ]:
# Cell 17: Create a Dataset from dict and save it

# Build from Python dicts
cs_dataset = Dataset.from_dict({
    "id": [c["id"] for c in cs_conversations],
    "conversations": [c["conversations"] for c in cs_conversations],
    "domain": [c["domain"] for c in cs_conversations],
})

print(f"Customer Service Dataset: {len(cs_dataset)} examples")
print(f"Features: {cs_dataset.features}")

# Add train/val split
cs_split = cs_dataset.train_test_split(test_size=0.15, seed=SEED)
print(f"\nTrain: {len(cs_split['train'])} examples")
print(f"Test: {len(cs_split['test'])} examples")

# Save to disk
cs_split.save_to_disk("./customer_service_dataset")
print("\nDataset saved to ./customer_service_dataset/")

# Also save as JSON for portability
cs_split["train"].to_json("./customer_service_train.json")
cs_split["test"].to_json("./customer_service_test.json")
print("JSON exports saved.")

In [ ]:
# Cell 18: Inspect our custom dataset

print("SAMPLE FROM OUR CUSTOMER SERVICE DATASET\n")
for idx in range(3):
    pretty_print_conversation(cs_split["train"][idx], index=f"cs_{idx}")

# Quick stats
turn_counts = [len(ex["conversations"]) for ex in cs_split["train"]]
word_counts = [sum(len(t["value"].split()) for t in ex["conversations"]) for ex in cs_split["train"]]
print(f"Avg turns per conversation: {np.mean(turn_counts):.1f}")
print(f"Avg words per conversation: {np.mean(word_counts):.1f}")
print(f"Multi-turn conversations: {sum(1 for t in turn_counts if t > 2)} / {len(turn_counts)}")

In [ ]:
# Cell 19: Summary and key takeaways

print("=" * 60)
print("NOTEBOOK 1 SUMMARY")
print("=" * 60)
print(f"""
We covered:
  - Loaded 3 datasets from Hugging Face Hub (Alpaca, Dolly, OpenOrca)
  - Converted between Alpaca, ShareGPT, and Qwen Chat formats
  - Built a cleaning pipeline: quality filtering + dedup
  - Demonstrated MinHash for near-duplicate detection
  - Created stratified train/val splits
  - Visualized data distributions with matplotlib
  - Human inspection script with quality flags
  - Built a custom 80-example customer-service dataset from scratch
  - Exported as HF Dataset and JSON

Key Functions You Can Reuse:
  - alpaca_to_sharegpt(), dolly_to_sharegpt(): Format converters
  - quality_filter(): Heuristic-based quality checks
  - dedup_exact(): Hash-based deduplication
  - pretty_print_conversation(): Human-readable inspection
  - Dataset.from_dict(): Create datasets from Python dicts
""")